<a href="https://colab.research.google.com/github/chuy-zip/PROYECTO2_DS/blob/main/PROY2_DS_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import pandas as pd
import numpy as np
from datasets import load_dataset
import torch.optim as optim
import time
from sklearn.metrics import accuracy_score

In [ ]:


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Cargar dataset IMDB de Hugging Face
print("Cargando dataset IMDB...")
dataset = load_dataset('imdb')

# Explorar el dataset
print("\n=== ESTRUCTURA DEL DATASET ===")
print(f"Splits disponibles: {list(dataset.keys())}")

for split in dataset.keys():
    print(f"\n{split.upper()}:")
    print(f"  Número de ejemplos: {len(dataset[split])}")
    print(f"  Columnas: {dataset[split].column_names}")
    print(f"  Características: {dataset[split].features}")

# Convertir a DataFrames para mejor visualización
print("\n=== CONVIRTIENDO A DATAFRAMES ===")
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Mostrar información clave
print("\n=== INFORMACIÓN CLAVE ===")
print("\nPrimeras 3 filas del training set:")
print(train_df[['label', 'text']].head(3))

print("\nDistribución de etiquetas en training:")
print(train_df['label'].value_counts().sort_index())

print("\nDistribución de etiquetas en test:")
print(test_df['label'].value_counts().sort_index())

print("\n=== EJEMPLOS DE TEXTO ===")
for i in range(2):
    sentiment = "POSITIVA" if train_df.iloc[i]['label'] == 1 else "NEGATIVA"
    print(f"\nEjemplo {i+1} - {sentiment}:")
    print(f"{train_df.iloc[i]['text'][:200]}...")

In [ ]:
print("=== TOKENIZACIÓN ===")

def simple_tokenizer(text):
    """Tokenizador básico para inglés"""
    # Convertir a minúsculas y dividir por espacios
    text = text.lower().replace('<br />', ' ')  # Limpiar tags HTML comunes en IMDB
    tokens = text.split()
    return tokens

# Probar el tokenizador
test_text = "This is a great movie! I loved it."
print(f"Texto original: {test_text}")
print(f"Tokens: {simple_tokenizer(test_text)}")

# Aplicar tokenización a todo el dataset
print("\nTokenizando el dataset...")
train_df['tokens'] = train_df['text'].apply(simple_tokenizer)
test_df['tokens'] = test_df['text'].apply(simple_tokenizer)

print(f"Ejemplo de tokens: {train_df['tokens'].iloc[0][:10]}...")

=== TOKENIZACIÓN ===
Texto original: This is a great movie! I loved it.
Tokens: ['this', 'is', 'a', 'great', 'movie!', 'i', 'loved', 'it.']

Tokenizando el dataset...
Ejemplo de tokens: ['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because']...


In [ ]:
print("\n=== CONSTRUCCIÓN DEL VOCABULARIO ===")

def build_vocab(token_lists, min_freq=5, max_vocab_size=45000):
    """Construye vocabulario con límite de tamaño"""
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    # Ordenar por frecuencia y limitar tamaño
    most_common = counter.most_common(max_vocab_size-2)  # -2 para <pad> y <unk>

    # Crear vocabulario
    vocab = {'<pad>': 0, '<unk>': 1}
    for idx, (word, count) in enumerate(most_common):
        if count >= min_freq:
            vocab[word] = idx + 2

    print(f"Vocabulario: {len(vocab)} palabras (límite: {max_vocab_size})")
    return vocab

# Construir vocabulario solo con datos de entrenamiento
vocab = build_vocab(train_df['tokens'])
vocab_size = len(vocab)

print(f"Tamaño del vocabulario: {vocab_size}")
print(f"Ejemplos de palabras en vocabulario: {list(vocab.items())[:10]}")

In [ ]:
print("\n=== CONVERSIÓN A ÍNDICES Y PADDING ===")

def tokens_to_indices(tokens_list, vocab, max_length=200):
    """Convierte tokens a índices y aplica padding"""
    sequences = []
    for tokens in tokens_list:
        # Convertir tokens a índices, usando <unk> para tokens no encontrados
        indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
        # Truncar si es muy largo
        indices = indices[:max_length]
        sequences.append(torch.tensor(indices, dtype=torch.long))

    # Aplicar padding
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=vocab['<pad>'])
    return padded_sequences

# Convertir a índices
max_sequence_length = 200
X_train = tokens_to_indices(train_df['tokens'], vocab, max_sequence_length)
X_test = tokens_to_indices(test_df['tokens'], vocab, max_sequence_length)

# Convertir labels a tensores
y_train = torch.tensor(train_df['label'].values, dtype=torch.long)
y_test = torch.tensor(test_df['label'].values, dtype=torch.long)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de y_train: {y_train.shape}")
print(f"Forma de X_test: {X_test.shape}")
print(f"Forma de y_test: {y_test.shape}")

In [ ]:
print("\n=== DIVISIÓN TRAIN/VALIDACIÓN ===")

# Crear dataset completo de entrenamiento
train_dataset = TensorDataset(X_train, y_train)

# Dividir 80% train, 20% validación
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # Para reproducibilidad
)

print(f"Ejemplos en entrenamiento: {len(train_dataset)}")
print(f"Ejemplos en validación: {len(val_dataset)}")
print(f"Ejemplos en prueba: {len(X_test)}")

In [ ]:
print("\n=== CREANDO DATALOADERS ===")

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

print(f"Número de batches en entrenamiento: {len(train_loader)}")
print(f"Número de batches en validación: {len(val_loader)}")
print(f"Número de batches en prueba: {len(test_loader)}")

### LTSM

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, num_classes=2, dropout=0.3):
        super(LSTMClassifier, self).__init__()

        # Capa de embeddings (igual que en RNN)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # Capa LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        # Capa final de clasificación
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_length)
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)
        lstm_out, (h_n, c_n) = self.lstm(embedded)  # lstm_out: (batch, seq, hidden_dim)

        last_hidden = h_n[-1, :, :]  # (batch_size, hidden_dim)

        output = self.dropout(last_hidden)
        output = self.fc(output)  # (batch_size, num_classes)
        return output

In [ ]:
print("\n=== INICIALIZANDO MODELO LSTM ===")

lstm_model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_classes=2
)

lstm_model = lstm_model.to(device)
lstm_params = count_parameters(lstm_model)
print(f"Número de parámetros entrenables del LSTM: {lstm_params:,}")
print(f"Arquitectura LSTM:\n{lstm_model}")

# Entrenar modelo LSTM (igual que el RNN)
lstm_results = train_model(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    model_name="LSTM"
)

# Evaluar en test
lstm_test_accuracy = evaluate_model(lstm_model, test_loader, "LSTM")
lstm_results['test_accuracy'] = lstm_test_accuracy